# 01 — Stim Basics: Measurements, Parity Checks, and Syndromes

## Goal

Build and interpret the smallest useful QEC experiment:

1. construct and sample simple Stim circuits;
2. measure the parity of two data qubits using an ancilla;
3. inject an $X$ error and observe the resulting syndrome.

**How to use this notebook:** run cells in order. Before running a cell marked **Prediction**, write down what you expect to see.

## 1. Import Stim

Stim simulates stabilizer circuits. This notebook assumes it was installed with:

```bash
uv add stim
```

In [ ]:
import stim
import numpy as np

print(f"Stim version: {stim.__version__}")

## 2. A deterministic measurement

Every Stim circuit starts with each qubit in $|0\rangle$.

**Prediction:** what should measuring qubit 0 return after applying an $X$ gate?

In [ ]:
x_circuit = stim.Circuit()
x_circuit.append("X", [0])
x_circuit.append("M", [0])

print(x_circuit)
print(x_circuit.compile_sampler().sample(shots=10))

The result should always be `True` (or 1):

$$|0\rangle \xrightarrow{X} |1\rangle,$$

and a computational-basis measurement of $|1\rangle$ is deterministic.

## 3. A probabilistic measurement

The Hadamard gate prepares $|+\rangle$ from $|0\rangle$.

**Prediction:** after 1,000 shots, roughly how many results should be 0 and how many should be 1?

In [ ]:
h_circuit = stim.Circuit()
h_circuit.append("H", [0])
h_circuit.append("M", [0])

samples = h_circuit.compile_sampler().sample(shots=1_000)
zeros = int(np.sum(samples[:, 0] == 0))
ones = int(np.sum(samples[:, 0] == 1))

print(h_circuit)
print(f"0 results: {zeros}")
print(f"1 results: {ones}")

Because $H|0\rangle = |+\rangle = (|0\rangle + |1\rangle)/\sqrt{2}$, a $Z$-basis measurement gives 0 and 1 with approximately equal probability.

The key Stim workflow is:

```text
construct circuit → compile sampler → sample many shots → interpret measurement record
```

## 4. Measure two-qubit parity with an ancilla

Qubits 0 and 1 are **data qubits**. Qubit 2 is an **ancilla**.

Two CNOTs write the parity of the data qubits into the ancilla:

| Data state | Parity | Ancilla measurement |
|---|---:|---:|
| $|00\rangle$ | even | 0 |
| $|01\rangle$ | odd | 1 |
| $|10\rangle$ | odd | 1 |
| $|11\rangle$ | even | 0 |

### Circuit diagram

![Ancilla-based two-qubit parity measurement](assets/parity_measurement.svg)

Read left to right:

1. $q_0$ controls the first CNOT, targeting ancilla $a$.
2. $q_1$ controls the second CNOT, also targeting $a$.
3. Measuring $a$ returns the parity bit: 0 for even parity and 1 for odd parity.

The data qubits are not measured by this circuit.

In [ ]:
def parity_measurement(data_bits):
    """Measure the parity of two computational-basis data qubits.

    Args:
        data_bits: a tuple such as (0, 0), (0, 1), (1, 0), or (1, 1).

    Returns:
        A Stim circuit that measures the ancilla only.
    """
    circuit = stim.Circuit()

    # Prepare |data_bits[0] data_bits[1]>.
    for qubit, bit in enumerate(data_bits):
        if bit:
            circuit.append("X", [qubit])

    # Copy parity contributions from each data qubit onto ancilla 2.
    circuit.append("CX", [0, 2])
    circuit.append("CX", [1, 2])

    # Read the ancilla: 0 means even parity, 1 means odd parity.
    circuit.append("M", [2])
    return circuit

In [ ]:
for data_bits in [(0, 0), (0, 1), (1, 0), (1, 1)]:
    circuit = parity_measurement(data_bits)
    measured_parity = int(circuit.compile_sampler().sample(shots=1)[0, 0])
    expected_parity = data_bits[0] ^ data_bits[1]
    print(
        f"data={data_bits}, measured={measured_parity}, "
        f"expected={expected_parity}"
    )

## 5. Why this is a stabilizer measurement

This circuit measures the Pauli observable $Z_0Z_1$.

- Even parity states $|00\rangle$ and $|11\rangle$ are $+1$ eigenstates.
- Odd parity states $|01\rangle$ and $|10\rangle$ are $-1$ eigenstates.
- Stim stores the measured eigenvalue as a bit: `0 ↔ +1`, `1 ↔ -1`.

The ancilla reveals **parity**, not the complete data state. For example, $|00\rangle$ and $|11\rangle$ yield the same ancilla result.

## 6. Inject an error and observe a syndrome

Treat $|00\rangle$ as a simple code state stabilized by $Z_0Z_1=+1$.

An $X$ error on exactly one data qubit anticommutes with $Z_0Z_1$, changing the measurement from $+1$ to $-1$. That change is a **syndrome**.

**Prediction:** which cases below give syndrome bit 1?

In [ ]:
def z_parity_syndrome(x_error_qubits=()):
    """Measure Z0 Z1 after optional X errors on data qubits 0 and 1."""
    circuit = stim.Circuit()

    for qubit in x_error_qubits:
        circuit.append("X", [qubit])

    circuit.append("CX", [0, 2])
    circuit.append("CX", [1, 2])
    circuit.append("M", [2])
    return circuit

cases = {
    "no error": (),
    "X error on qubit 0": (0,),
    "X error on qubit 1": (1,),
    "X errors on both qubits": (0, 1),
}

for label, error_qubits in cases.items():
    circuit = z_parity_syndrome(error_qubits)
    syndrome_bit = int(circuit.compile_sampler().sample(shots=1)[0, 0])
    print(f"{label:25} → syndrome bit {syndrome_bit}")

Expected interpretation:

| Case | Stabilizer eigenvalue | Syndrome bit |
|---|---:|---:|
| No error | $+1$ | 0 |
| $X_0$ | $-1$ | 1 |
| $X_1$ | $-1$ | 1 |
| $X_0X_1$ | $+1$ | 0 |

One parity check detects that an odd number of $X$ errors occurred, but it cannot distinguish whether the error was on qubit 0 or qubit 1. A full QEC code uses several stabilizer checks so that the combined syndrome locates an error class.

## 7. Checkpoint

Before moving on, answer these in your own words:

1. What is the role of the ancilla qubit?
2. Why do $|00\rangle$ and $|11\rangle$ produce the same parity result?
3. Why does a single $X$ error flip the $Z_0Z_1$ stabilizer outcome?
4. Why can this one syndrome measurement not tell whether the error was $X_0$ or $X_1$?
5. What additional information would be needed to identify the error location?

When these answers feel clear, the next notebook will construct the three-qubit repetition code with two parity checks: $Z_0Z_1$ and $Z_1Z_2$.